# Fast audio transcription (CSV / DOCX)

Upload an audio file, get a transcript, download it as **CSV** (with timestamps) or **DOCX**.

## Which engine is fastest?

YouTube's own captioning model (Google's Universal Speech Model, USM) is **not downloadable**; its public form is the paid *Chirp* model inside Google Cloud Speech-to-Text. No engine, including Google's, transcribes a 1-2 hour file in milliseconds. Realistic speeds for a **1 hour** file on this machine (8 CPU cores, no NVIDIA GPU):

| Engine | 1 hour of audio takes | Needs |
|---|---|---|
| **Cloud: Groq Whisper large-v3-turbo** | about 30-90 seconds | free API key from console.groq.com, internet |
| **Local: faster-whisper `small` (int8, batched)** | about 5-10 minutes | nothing, model already cached |
| Local: faster-whisper `base` | about 3-5 minutes | nothing, cached, lower accuracy |
| Local: faster-whisper `large-v3-turbo` | 20-40 minutes | 1.6 GB download, free disk space |
| Old: openai-whisper `turbo` on CPU | 40-90 minutes | what the notebook used before |

The cloud engine is the only way to get near-instant results for long files. The local engine is the fastest fully offline option and needs no download.

## Before you start

- Run the install cell once. Everything it lists is already installed here, so it finishes instantly.
- FFmpeg must be on PATH (`winget install Gyan.FFmpeg`).
- For the cloud engine, set the environment variable `GROQ_API_KEY` or paste the key into the app's key box.
- **Free up disk space.** The C: drive is currently full; the app deletes its intermediate audio, but it still needs room for the exports.


In [ ]:
%pip install -q faster-whisper httpx gradio python-docx


## Transcription and export helpers

Two engines share one export path:

- **Local** uses faster-whisper (CTranslate2) with int8 weights, the batched inference pipeline and voice-activity detection, which skips silence. This is 4-8x faster than the original openai-whisper code on the same CPU.
- **Cloud** converts the audio to 16 kHz mono FLAC, splits it into 10-minute chunks (Groq's free tier accepts files up to 25 MB), sends the chunks to Groq's Whisper endpoint and stitches the timestamps back together.

Intermediate audio is deleted as soon as it has been transcribed.


In [ ]:
import csv
import os
import shutil
import subprocess
import tempfile
import time
import uuid
from functools import lru_cache
from pathlib import Path

import gradio as gr
import httpx
import numpy as np
from docx import Document
from faster_whisper import BatchedInferencePipeline, WhisperModel

LOCAL_MODELS = ["small", "base", "tiny", "medium", "large-v3-turbo", "large-v3"]
CLOUD_MODELS = ["whisper-large-v3-turbo", "whisper-large-v3"]
ENGINES = ["Local (faster-whisper, offline)", "Cloud (Groq Whisper, fastest)"]

# CTranslate2 runs best on physical cores; more threads mostly add memory use.
CPU_THREADS = max(1, min(4, (os.cpu_count() or 4)))
GROQ_URL = "https://api.groq.com/openai/v1/audio/transcriptions"
CHUNK_SECONDS = 600  # 10-minute FLAC chunks stay well under Groq's 25 MB limit

FFMPEG = shutil.which("ffmpeg")
if FFMPEG is None:
    raise RuntimeError(
        "FFmpeg was not found on PATH. Install FFmpeg, restart the notebook kernel, and run this cell again."
    )

print(f"Local engine: faster-whisper on CPU, int8, {CPU_THREADS} threads")


@lru_cache(maxsize=1)
def load_local_model(model_name):
    """Load and cache one CTranslate2 Whisper model (int8 on CPU).

    Only one model is kept in memory; switching models frees the previous one.
    """
    if model_name not in LOCAL_MODELS:
        raise ValueError(f"Unsupported model: {model_name}")
    model = WhisperModel(
        model_name,
        device="cpu",
        compute_type="int8",
        cpu_threads=CPU_THREADS,
    )
    return BatchedInferencePipeline(model=model)


def run_ffmpeg(args):
    try:
        subprocess.run([FFMPEG, "-nostdin", "-y", *args], check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as error:
        details = error.stderr.strip().splitlines()[-1] if error.stderr.strip() else "FFmpeg failed."
        raise gr.Error(f"Could not read audio from the uploaded file: {details}") from error


def load_pcm(audio_path, work_dir):
    """Decode any audio/video file to 16 kHz mono float32 samples via FFmpeg.

    faster-whisper's own decoder breaks with the PyAV version installed here, so the
    raw PCM is read with NumPy instead and the temporary file is removed right away.
    """
    pcm_path = work_dir / "audio.s16le"
    run_ffmpeg(["-i", str(audio_path), "-vn", "-ac", "1", "-ar", "16000", "-f", "s16le", "-c:a", "pcm_s16le", str(pcm_path)])
    samples = np.fromfile(pcm_path, dtype=np.int16).astype(np.float32) / 32768.0
    pcm_path.unlink(missing_ok=True)
    if samples.size == 0:
        raise gr.Error("FFmpeg produced no audio from this file.")
    return samples


def transcribe_local(audio_path, model_name, work_dir, progress):
    samples = load_pcm(audio_path, work_dir)

    progress(0.05, desc=f"Loading {model_name} model")
    pipeline = load_local_model(model_name)
    total = max(samples.size / 16000.0, 1.0)

    # Bigger batches are faster but need more RAM. On a low-memory machine the
    # encoder can fail with "mkl_malloc: failed to allocate memory"; when that
    # happens the whole pass is retried with a smaller batch.
    last_error = None
    for batch_size in (8, 4, 2, 1):
        try:
            segments_iter, _info = pipeline.transcribe(
                samples,
                batch_size=batch_size,
                vad_filter=True,
                beam_size=1,
                word_timestamps=False,
            )
            segments = []
            for segment in segments_iter:
                segments.append({"start": segment.start, "end": segment.end, "text": segment.text})
                progress(
                    min(0.05 + 0.9 * segment.end / total, 0.95),
                    desc=f"Transcribing {int(segment.end // 60)} min (batch {batch_size})",
                )
            return segments
        except (RuntimeError, MemoryError) as error:
            if not any(word in str(error).lower() for word in ("allocate", "memory", "alloc")):
                raise
            last_error = error
            print(f"Low memory at batch size {batch_size}; retrying with a smaller batch.")
    raise gr.Error(
        "Not enough free RAM to run the local model. Close other programs (or stray Python servers) "
        f"and try again, or use the Cloud engine. Last error: {last_error}"
    )


def transcribe_cloud(audio_path, model_name, api_key, work_dir, progress):
    api_key = (api_key or os.environ.get("GROQ_API_KEY", "")).strip()
    if not api_key:
        raise gr.Error("Enter a Groq API key (or set GROQ_API_KEY) to use the cloud engine.")

    chunk_pattern = work_dir / "chunk_%04d.flac"
    run_ffmpeg([
        "-i", str(audio_path), "-vn", "-ac", "1", "-ar", "16000",
        "-f", "segment", "-segment_time", str(CHUNK_SECONDS), "-c:a", "flac", str(chunk_pattern),
    ])
    chunks = sorted(work_dir.glob("chunk_*.flac"))
    if not chunks:
        raise gr.Error("FFmpeg produced no audio from this file.")

    segments = []
    offset = 0.0
    headers = {"Authorization": f"Bearer {api_key}"}
    with httpx.Client(timeout=httpx.Timeout(300.0)) as client:
        for index, chunk in enumerate(chunks):
            progress(0.05 + 0.9 * index / len(chunks), desc=f"Uploading chunk {index + 1}/{len(chunks)}")
            with chunk.open("rb") as handle:
                response = client.post(
                    GROQ_URL,
                    headers=headers,
                    data={"model": model_name, "response_format": "verbose_json", "temperature": "0"},
                    files={"file": (chunk.name, handle, "audio/flac")},
                )
            chunk.unlink(missing_ok=True)
            if response.status_code != 200:
                raise gr.Error(f"Groq returned HTTP {response.status_code}: {response.text[:300]}")
            payload = response.json()
            for segment in payload.get("segments", []):
                segments.append({
                    "start": offset + float(segment["start"]),
                    "end": offset + float(segment["end"]),
                    "text": segment.get("text", ""),
                })
            offset += float(payload.get("duration") or CHUNK_SECONDS)
    return segments


def format_timestamp(seconds):
    seconds = int(seconds)
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"


def write_exports(segments, output_dir):
    csv_path = output_dir / "transcript.csv"
    with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["start_seconds", "end_seconds", "start", "end", "text"])
        for segment in segments:
            writer.writerow([
                round(segment["start"], 2),
                round(segment["end"], 2),
                format_timestamp(segment["start"]),
                format_timestamp(segment["end"]),
                segment["text"].strip(),
            ])

    docx_path = output_dir / "transcript.docx"
    document = Document()
    document.add_heading("Audio Transcript", level=1)
    for segment in segments:
        stamp = f"[{format_timestamp(segment['start'])} - {format_timestamp(segment['end'])}]"
        document.add_paragraph(f"{stamp} {segment['text'].strip()}")
    document.save(docx_path)
    return csv_path, docx_path


def transcribe_audio(audio_path, engine, local_model, cloud_model, api_key, progress=gr.Progress()):
    """Transcribe an audio file and return transcript text, timing note, CSV path and DOCX path."""
    if not audio_path:
        raise gr.Error("Upload an audio file before starting transcription.")
    if not Path(audio_path).is_file():
        raise gr.Error("The uploaded audio file could not be found.")

    output_dir = Path(tempfile.gettempdir()) / f"audio_transcript_{uuid.uuid4().hex}"
    output_dir.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    progress(0.0, desc="Converting audio")

    if engine.startswith("Cloud"):
        segments = transcribe_cloud(audio_path, cloud_model, api_key, output_dir, progress)
        used = f"Groq {cloud_model}"
    else:
        segments = transcribe_local(audio_path, local_model, output_dir, progress)
        used = f"faster-whisper {local_model} (CPU int8)"

    segments = [s for s in segments if s["text"].strip()]
    if not segments:
        raise gr.Error("No speech was detected in the uploaded file.")

    transcript = " ".join(s["text"].strip() for s in segments)
    csv_path, docx_path = write_exports(segments, output_dir)

    elapsed = time.perf_counter() - started
    audio_len = segments[-1]["end"]
    note = f"{used}: {format_timestamp(audio_len)} of audio in {elapsed:.1f} s ({audio_len / max(elapsed, 0.01):.0f}x real time)"
    return transcript, note, str(csv_path), str(docx_path)


## Launch the Gradio app

Run the install cell and the helpers cell first. `share=False` keeps the app local to your machine.


In [ ]:
required_setup = ("gr", "ENGINES", "transcribe_audio")
missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError(
        "Setup is incomplete. Run the install cell and the helpers cell before launching the app. "
        f"Missing: {', '.join(missing_setup)}"
    )

with gr.Blocks(title="Audio Transcriber") as demo:
    gr.Markdown(
        "# Audio Transcriber\n"
        "Upload an audio file, transcribe it, download CSV or DOCX. "
        "**Cloud** is fastest for long files (needs a Groq key); **Local** works offline."
    )
    with gr.Row():
        audio_input = gr.Audio(label="Audio file", sources=["upload"], type="filepath")
        with gr.Column():
            engine_input = gr.Radio(choices=ENGINES, value=ENGINES[0], label="Engine")
            local_model_input = gr.Dropdown(
                choices=LOCAL_MODELS,
                value="small",
                label="Local model",
                info="small is the best already-downloaded model; medium and larger must download first.",
            )
            cloud_model_input = gr.Dropdown(
                choices=CLOUD_MODELS,
                value="whisper-large-v3-turbo",
                label="Cloud model",
                visible=False,
            )
            api_key_input = gr.Textbox(
                label="Groq API key",
                type="password",
                placeholder="gsk_... (or set GROQ_API_KEY)",
                visible=False,
            )

    def toggle_engine(engine):
        cloud = engine.startswith("Cloud")
        return (
            gr.update(visible=not cloud),
            gr.update(visible=cloud),
            gr.update(visible=cloud),
        )

    engine_input.change(
        toggle_engine,
        inputs=engine_input,
        outputs=[local_model_input, cloud_model_input, api_key_input],
    )

    transcribe_button = gr.Button("Transcribe", variant="primary")
    timing_output = gr.Markdown()
    transcript_output = gr.Textbox(label="Transcript", lines=16)
    with gr.Row():
        csv_output = gr.File(label="Download CSV")
        docx_output = gr.File(label="Download DOCX")
    transcribe_button.click(
        fn=transcribe_audio,
        inputs=[audio_input, engine_input, local_model_input, cloud_model_input, api_key_input],
        outputs=[transcript_output, timing_output, csv_output, docx_output],
    )

demo.launch(share=False, server_port=8002, show_error=True)
